# Gemini 02. API 비용 계산과 예상 비용 확인

Claude 06 노트북과 같은 흐름이다. **요청 전 예상 → 요청 후 계산 → 원화로 보기**

## 먼저 알아둘 것

- **Gemini API도 단가를 알려주는 API가 없다.** 01의 모델 목록 조회에도 가격은 없다.
- **OpenRouter 단가를 그대로 쓰면 안 된다.** Claude는 OpenRouter 단가가 공식 단가와 같았지만, Gemini는 다른 경우가 있다. (2026-10-19 확인: Gemini 3.8 Flash 공식 $0.75 / $3.75, OpenRouter $0.825 / $4.125, 약 10% 높음)
  → 그래서 이 노트북은 **[공식 가격 페이지](https://ai.google.dev/gemini-api/docs/pricing)를 보고 직접 관리하는 단가표**를 기본으로 쓰고, OpenRouter는 비교용으로만 쓴다.
- **무료 등급(Free tier)이면 실제 비용은 0이다.** 이 노트북의 금액은 유료 등급(Tier 1 이상) 기준이다. 현재 등급은 [AI Studio Projects](https://aistudio.google.com/projects)에서 확인한다.
- Claude보다 단가 구조가 복잡하다.
  - 일부 모델(Pro)은 **입력이 20만 토큰을 넘으면** 단가가 오른다.
  - 일부 모델은 **현재 가격에 기한이 있고, 그 이후 2배로 오른다.** (Gemini 4 Argon은 2026-12-01, Gemini 3.8/3.6 Flash는 2027-01-01부터) → 아래 "가격 변경 일정" 참고
  - 입력 종류(텍스트, 이미지, 오디오)마다 단가가 다를 수 있다. 이 노트북은 **텍스트 기준**이다.
- 출력 요금에는 **thinking 토큰이 포함**된다. `usage_metadata`에서는 답변(`candidates_token_count`)과 thinking(`thoughts_token_count`)이 따로 나오므로 더해서 계산한다.

In [1]:
from datetime import date

import requests
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()

MODEL = "gemini-3.8-flash"

## 1. 단가표 (공식 가격 페이지 기준, 직접 관리)

100만 토큰당 USD, 텍스트 입력 기준, 2026-10-19에 공식 페이지에서 확인한 값이다.

- `tiers`: 입력 토큰이 `over` 값을 **넘으면** 적용되는 단가
- `changes`: 공지된 가격 변경. `from` 날짜부터 적용

**가격은 바뀔 수 있으므로, 중요한 계산 전에는 공식 페이지와 다시 대조한다.**

In [2]:
PRICING_CHECKED = "2026-10-19"

PRICING = {
    "gemini-3.8-flash": {
        "input": 0.75, "output": 3.75, "cache_read": 0.075,
        "changes": [{"from": date(2027, 1, 1), "input": 1.50, "output": 7.50, "cache_read": 0.15}],
    },
    "gemini-3.6-flash": {
        "input": 0.75, "output": 3.75, "cache_read": 0.075,
        "changes": [{"from": date(2027, 1, 1), "input": 1.50, "output": 7.50, "cache_read": 0.15}],
    },
    "gemini-4-argon": {  # 무료 등급 없음, 암묵적 캐싱만 지원
        "input": 2.00, "output": 10.00, "cache_read": 0.10,
        "changes": [{"from": date(2026, 12, 1), "input": 4.00, "output": 20.00, "cache_read": 0.20}],
    },
    "gemini-3.1-flash-lite": {"input": 0.25, "output": 1.50, "cache_read": 0.025},
    "gemini-3.1-pro-preview": {
        "input": 2.00, "output": 12.00, "cache_read": 0.20,
        "tiers": [{"over": 200_000, "input": 4.00, "output": 18.00, "cache_read": 0.40}],
    },
    "gemini-2.5-flash": {"input": 0.30, "output": 2.50, "cache_read": 0.03},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40, "cache_read": 0.01},
    "gemini-2.5-pro": {
        "input": 1.25, "output": 10.00, "cache_read": 0.125,
        "tiers": [{"over": 200_000, "input": 2.50, "output": 15.00, "cache_read": 0.25}],
    },
}


def get_rates(model: str, prompt_tokens: int, on: date | None = None) -> dict:
    """모델, 입력 토큰 수, 날짜에 맞는 단가(100만 토큰당 USD)를 반환"""
    model = model.removeprefix("models/")
    if model not in PRICING:
        raise KeyError(f"단가표에 없는 모델: {model} (PRICING에 추가 필요)")
    p = PRICING[model]
    on = on or date.today()

    rates = {k: p[k] for k in ("input", "output", "cache_read")}
    for change in p.get("changes", []):  # 공지된 가격 변경 반영
        if on >= change["from"]:
            rates = {k: change[k] for k in ("input", "output", "cache_read")}
    for tier in p.get("tiers", []):  # 긴 프롬프트 구간 반영
        if prompt_tokens > tier["over"]:
            rates = {k: tier[k] for k in ("input", "output", "cache_read")}
    return rates


print(f"단가표 확인일: {PRICING_CHECKED}\n")
print(f"{'Model':<24} {'Input':>7} {'Output':>7} {'Cache':>7}   (USD / 1M tokens, 오늘 기준)")
print("-" * 70)
for m in PRICING:
    r = get_rates(m, 0)
    note = "  * 20만 토큰 초과 시 인상" if PRICING[m].get("tiers") else ""
    if PRICING[m].get("changes"):
        note += f"  * {PRICING[m]['changes'][0]['from']}부터 인상"
    print(f"{m:<24} {r['input']:>7.3f} {r['output']:>7.2f} {r['cache_read']:>7.3f}{note}")

단가표 확인일: 2026-10-19

Model                      Input  Output   Cache   (USD / 1M tokens, 오늘 기준)
----------------------------------------------------------------------
gemini-3.8-flash           0.750    3.75   0.075  * 2027-01-01부터 인상
gemini-3.6-flash           0.750    3.75   0.075  * 2027-01-01부터 인상
gemini-4-argon             2.000   10.00   0.100  * 2026-12-01부터 인상
gemini-3.1-flash-lite      0.250    1.50   0.025
gemini-3.1-pro-preview     2.000   12.00   0.200  * 20만 토큰 초과 시 인상
gemini-2.5-flash           0.300    2.50   0.030
gemini-2.5-flash-lite      0.100    0.40   0.010
gemini-2.5-pro             1.250   10.00   0.125  * 20만 토큰 초과 시 인상


### 가격 변경 일정 (기한이 있는 현재 가격)

공식 가격 페이지에는 일부 모델의 현재 가격이 **"~까지(through)"** 기한과 함께 적혀 있고, 그 이후 가격이 **"~부터(starting)"** 로 따로 적혀 있다. (2026-10-19 확인)

| 모델 | 현재 (입력 / 출력) | 변경 후 | 변경일 |
|---|---|---|---|
| Gemini 4 Argon | $2.00 / $10.00 | $4.00 / $20.00 | **2026-12-01** |
| Gemini 3.8 Flash | $0.75 / $3.75 | $1.50 / $7.50 | 2027-01-01 |
| Gemini 3.6 Flash | $0.75 / $3.75 | $1.50 / $7.50 | 2027-01-01 |

- 공식 페이지는 현재 가격을 "할인"이나 "프로모션"이라고 표현하지는 않는다. 다만 **기한이 지나면 2배가 된다는 점은 명시**되어 있다.
- 캐시 단가, Batch, Flex, Priority 단가도 함께 오른다.
- 이 표에 없는 모델(3.1 Pro, 3.1 Flash-Lite 등)은 기한 표시가 없지만, **가격은 언제든 바뀔 수 있다.** 장기 비용을 계산할 때는 공식 페이지를 다시 확인한다.

**비용 계획에 주는 영향**
- 지금 측정한 비용으로 몇 달 뒤 예산을 잡으면 **실제로는 2배가 나올 수 있다.** 장기 예산은 변경 후 가격 기준으로 잡는 것이 안전하다.
- 위 단가표의 `get_rates(model, tokens, on=날짜)`로 특정 날짜 기준 단가를 구할 수 있다.

In [3]:
today = date.today()

print(f"{'Model':<20} {'Now In/Out':>14} {'After In/Out':>15} {'Change date':>12} {'Days left':>10}")
print("-" * 76)
for m, p in PRICING.items():
    for change in p.get("changes", []):
        now = get_rates(m, 0, on=today)
        after = get_rates(m, 0, on=change["from"])
        days_left = (change["from"] - today).days
        print(f"{m:<20} {now['input']:>6.2f}/{now['output']:<7.2f} {after['input']:>6.2f}/{after['output']:<8.2f} "
              f"{str(change['from']):>12} {days_left:>10}")

Model                    Now In/Out    After In/Out  Change date  Days left
----------------------------------------------------------------------------
gemini-3.8-flash       0.75/3.75      1.50/7.50       2027-01-01         74
gemini-3.6-flash       0.75/3.75      1.50/7.50       2027-01-01         74
gemini-4-argon         2.00/10.00     4.00/20.00      2026-12-01         43


### (참고) OpenRouter 단가와 비교

Claude 06에서 쓴 OpenRouter 공개 API로 같은 모델의 단가를 가져와 공식 단가와 비교한다. (무료, 키 불필요)
차이가 있으면 **공식 단가가 기준**이다. OpenRouter를 통해 쓸 때의 가격일 뿐, Google에 직접 요청할 때의 가격이 아니다.

In [4]:
try:
    r = requests.get("https://openrouter.ai/api/v1/models", timeout=30)
    r.raise_for_status()
    openrouter = {m["id"]: m["pricing"] for m in r.json()["data"]}
except (requests.RequestException, ValueError, KeyError) as e:
    openrouter = {}
    print(f"OpenRouter 조회 실패: {e}")

print(f"{'Model':<24} {'Official In/Out':<16} {'OpenRouter In/Out':<20} Diff")
print("-" * 72)
for m in PRICING:
    official = get_rates(m, 0)
    p = openrouter.get(f"google/{m}")
    if p is None:
        print(f"{m:<24} {official['input']:>7.3f}/{official['output']:<8.2f} {'(없음)':>18}")
        continue
    o_in, o_out = float(p["prompt"]) * 1e6, float(p["completion"]) * 1e6
    diff = "같음" if (abs(o_in - official["input"]) < 1e-9 and abs(o_out - official["output"]) < 1e-9) else f"{(o_in / official['input'] - 1) * 100:+.0f}%"
    print(f"{m:<24} {official['input']:>7.3f}/{official['output']:<8.2f} {o_in:>9.3f}/{o_out:<10.3f} {diff}")

Model                    Official In/Out  OpenRouter In/Out    Diff
------------------------------------------------------------------------
gemini-3.8-flash           0.750/3.75         0.825/4.125      +10%
gemini-3.6-flash           0.750/3.75         0.750/3.750      같음
gemini-4-argon             2.000/10.00        2.000/10.000     같음
gemini-3.1-flash-lite      0.250/1.50         0.250/1.500      같음
gemini-3.1-pro-preview     2.000/12.00        2.000/12.000     같음
gemini-2.5-flash           0.300/2.50         0.300/2.500      같음
gemini-2.5-flash-lite      0.100/0.40         0.100/0.400      같음
gemini-2.5-pro             1.250/10.00        1.250/10.000     같음


## 2. 비용 계산 함수

```
비용 = (입력 − 캐시) × 입력 단가
     + 캐시          × 캐시 단가
     + (답변 + thinking) × 출력 단가
```

`prompt_token_count`에는 캐시된 토큰이 **포함**되어 있으므로 빼고 계산한다. (Claude는 처음부터 나뉘어 있었음)

In [5]:
def calc_cost(model: str, usage) -> dict:
    """usage_metadata로 비용(USD)을 항목별로 계산"""
    prompt = usage.prompt_token_count or 0
    cached = usage.cached_content_token_count or 0
    answer = usage.candidates_token_count or 0
    thinking = usage.thoughts_token_count or 0

    r = get_rates(model, prompt)
    cost = {
        "input": (prompt - cached) * r["input"],
        "cache": cached * r["cache_read"],
        "answer": answer * r["output"],
        "thinking": thinking * r["output"],
    }
    cost = {k: v / 1_000_000 for k, v in cost.items()}
    cost["total"] = sum(cost.values())
    return cost

## 3. 요청 전: 예상 비용 확인

- **입력 토큰**: `client.models.count_tokens()`로 정확히 알 수 있다. **요금이 발생하지 않는다.**
- **출력 토큰**: 예상치로 추정하거나, 최대치로 상한을 본다.
- **최대치는 `max_output_tokens` 기준이다.** 테스트해 보니 Gemini는 **thinking도 `max_output_tokens` 한도 안에 포함**된다. (한도 200으로 요청 → thinking 188 + 답변 8에서 잘림) 그래서 `입력 + max_output_tokens × 출력 단가`가 그 요청의 비용 상한이다.
- `max_output_tokens`를 지정하지 않으면 모델 최대치(Gemini 3.8 Flash는 65,536)까지 쓸 수 있다.

In [6]:
MAX_OUTPUT_TOKENS = 8_000


def estimate_cost(model: str, contents, expected_output_tokens: int,
                  max_output_tokens: int = MAX_OUTPUT_TOKENS, on: date | None = None) -> dict:
    """요청 전 예상 비용. 입력은 count_tokens(무료)로 정확히, 출력은 예상치/최대치
    on: 이 날짜 기준 단가로 계산 (생략하면 오늘)"""
    count = client.models.count_tokens(model=model, contents=contents)
    r = get_rates(model, count.total_tokens, on=on)
    input_cost = count.total_tokens * r["input"] / 1_000_000
    return {
        "input_tokens": count.total_tokens,
        "expected": input_cost + expected_output_tokens * r["output"] / 1_000_000,
        "worst": input_cost + max_output_tokens * r["output"] / 1_000_000,
    }


def usd(x: float) -> str:
    return f"${x:.5f}"


prompt = "파이썬의 데코레이터가 무엇인지 예제 코드 1개와 함께 500자 이내로 설명해줘."

# 답변 500자 ≈ 500토큰 + thinking 여유 1,000토큰
est = estimate_cost(MODEL, prompt, expected_output_tokens=1_500)
print(f"모델             : {MODEL}")
print(f"입력 토큰 (정확) : {est['input_tokens']:,}")
print(f"예상 비용        : {usd(est['expected'])}   (출력 1,500 토큰 가정)")
print(f"최대 비용        : {usd(est['worst'])}   (max_output_tokens {MAX_OUTPUT_TOKENS:,} 전부 사용 시)")

모델             : gemini-3.8-flash
입력 토큰 (정확) : 32
예상 비용        : $0.00565   (출력 1,500 토큰 가정)
최대 비용        : $0.03002   (max_output_tokens 8,000 전부 사용 시)


### 같은 요청을 모델별로 비교 (무료)

In [7]:
compare_models = ["gemini-3.8-flash", "gemini-3.1-flash-lite", "gemini-3.1-pro-preview", "gemini-4-argon"]
AFTER = date(2027, 1, 1)  # 공지된 가격 변경이 모두 반영된 시점

print(f"{'Model':<24} {'Input tokens':>12} {'Expected':>12} {'Worst':>12} {'Exp. 2027~':>12}")
print("-" * 76)
for m in compare_models:
    e = estimate_cost(m, prompt, expected_output_tokens=1_500)
    later = estimate_cost(m, prompt, expected_output_tokens=1_500, on=AFTER)
    print(f"{m:<24} {e['input_tokens']:>12,} {usd(e['expected']):>12} {usd(e['worst']):>12} {usd(later['expected']):>12}")

Model                    Input tokens     Expected        Worst   Exp. 2027~
----------------------------------------------------------------------------
gemini-3.8-flash                   32     $0.00565     $0.03002     $0.01130
gemini-3.1-flash-lite              32     $0.00226     $0.01201     $0.00226
gemini-3.1-pro-preview             32     $0.01806     $0.09606     $0.01806
gemini-4-argon                     58     $0.01512     $0.08012     $0.03023


> - `Exp. 2027~`: 공지된 가격 변경이 모두 적용된 2027-01-01 기준 예상 비용. Argon과 3.8 Flash는 2배가 된다. 테스트에서는 Argon이 오늘 기준으로 3.1 Pro보다 쌌지만, 2027년 기준으로는 더 비싸졌다.
> - **같은 문장이라도 모델마다 입력 토큰 수가 다를 수 있다.** 테스트에서는 같은 프롬프트가 Gemini 3.x에서는 32토큰, Gemini 4 Argon에서는 58토큰으로 계산되었다(약 1.8배). 모델을 비교할 때 단가만 보지 말고 토큰 수도 함께 봐야 한다.

## 4. 요청 후: 실제 사용량으로 비용 계산

> 이 셀은 요청을 1회 보내므로 소액의 요금이 발생한다. (무료 등급이면 무료)

In [8]:
response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(max_output_tokens=MAX_OUTPUT_TOKENS),
)

u = response.usage_metadata
cost = calc_cost(response.model_version, u)  # 실제로 응답한 모델 기준
finish = response.candidates[0].finish_reason if response.candidates else None

print(f"입력      : {u.prompt_token_count or 0:>6,}  → {usd(cost['input'] + cost['cache'])}")
print(f"답변      : {u.candidates_token_count or 0:>6,}  → {usd(cost['answer'])}")
print(f"thinking  : {u.thoughts_token_count or 0:>6,}  → {usd(cost['thinking'])}")
print("-" * 36)
print(f"합계      :         → {usd(cost['total'])}")
print(f"\n종료 이유 : {finish}  (MAX_TOKENS면 한도에 걸려 잘린 것)")

입력      :     32  → $0.00002
답변      :    205  → $0.00077
thinking  :    666  → $0.00250
------------------------------------
합계      :         → $0.00329

종료 이유 : FinishReason.STOP  (MAX_TOKENS면 한도에 걸려 잘린 것)


In [9]:
output_tokens = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
print(f"예상 출력 토큰 : {1_500:>6,}    실제 : {output_tokens:>6,}")
print(f"예상 비용      : {usd(est['expected'])}   실제 : {usd(cost['total'])}")
print(f"최대 비용      : {usd(est['worst'])}   (실제 비용은 항상 이 이하)")

예상 출력 토큰 :  1,500    실제 :    871
예상 비용      : $0.00565   실제 : $0.00329
최대 비용      : $0.03002   (실제 비용은 항상 이 이하)


> 예상과 실제의 차이를 줄이는 방법은 Claude 06 노트북의 "예상과 실제의 차이를 줄이려면"과 같다. Gemini는 thinking 비중이 큰 편이라 thinking 양에 따라 차이가 더 크게 날 수 있다. thinking 양은 `GenerateContentConfig`의 thinking 설정으로 조절할 수 있다.

## 부록: 원화(KRW)로 보기

Claude 06 노트북의 부록과 같은 방식이다. 기준환율이므로 실제 카드 청구액(카드사 환율, 해외 결제 수수료, 세금)과는 다르다.

In [10]:
FALLBACK_USD_KRW = 1332.1  # 두 API 모두 실패 시 (2026-10-16 ECB 기준환율)


def get_usd_krw() -> tuple[float, str]:
    sources = [
        ("Frankfurter (ECB)", "https://api.frankfurter.dev/v1/latest?base=USD&symbols=KRW",
         lambda body: (body["rates"]["KRW"], body["date"])),
        ("ExchangeRate-API", "https://open.er-api.com/v6/latest/USD",
         lambda body: (body["rates"]["KRW"], body["time_last_update_utc"][:16])),
    ]
    for name, url, parse in sources:
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            rate, when = parse(r.json())
            return float(rate), f"{name}, {when}"
        except (requests.RequestException, ValueError, KeyError) as e:
            print(f"{name} 조회 실패 → 다음 출처 시도 ({e})")
    return FALLBACK_USD_KRW, "고정값 (2026-10-16)"


USD_KRW, RATE_SOURCE = get_usd_krw()


def krw(usd_amount: float) -> str:
    won = usd_amount * USD_KRW
    return f"{won:,.0f}원" if won >= 1 else f"{won:.2f}원"


print(f"환율: 1 USD = {USD_KRW:,.2f} KRW  ({RATE_SOURCE})\n")

# 한글 라벨은 화면 2칸이므로 정렬하지 않고, 원화 열 머리글은 1칸 넓게
rows = [("예상 비용", est["expected"]), ("최대 비용", est["worst"]), ("실제 비용", cost["total"])]
print(f"{'':<9}  {'USD':>10}  {'KRW':>11}")
print("-" * 36)
for label, amount in rows:
    print(f"{label}  {usd(amount):>10}  {krw(amount):>10}")

print()
for n in (100, 1_000):
    print(f"이 요청을 {n:,}번 보내면 약 {krw(cost['total'] * n)}")

환율: 1 USD = 1,332.10 KRW  (Frankfurter (ECB), 2026-10-16)

                  USD          KRW
------------------------------------
예상 비용    $0.00565          8원
최대 비용    $0.03002         40원
실제 비용    $0.00329          4원

이 요청을 100번 보내면 약 438원
이 요청을 1,000번 보내면 약 4,383원


## 정리

| 하고 싶은 것 | Claude | Gemini |
|---|---|---|
| 단가 출처 | OpenRouter (공식과 일치) | **공식 페이지 직접 관리** (OpenRouter와 다를 수 있음) |
| 입력 토큰 미리 세기 (무료) | `messages.count_tokens()` | `models.count_tokens()` |
| 출력 토큰 | `output_tokens` | `candidates_token_count` + `thoughts_token_count` |
| 비용 상한 | `max_tokens` 기준 | `max_output_tokens` 기준 (thinking 포함) |
| 특이사항 | Haiku 5.5 10만 토큰 초과 구간 | Pro 20만 토큰 초과 구간, 무료 등급 |
| 가격 변경 일정 | (현재 공지 없음) | **Argon 2026-12-01, 3.8/3.6 Flash 2027-01-01부터 2배** |